# ChartNarrator Stage 2 zero-shot VLM inference

This notebook evaluates the base `Qwen/Qwen2-VL-7B-Instruct` model without any LoRA adapter. It uses the same WA >= 4.5 VLM test split as the main Stage 2 experiment.

Original run record from the cleaned notebook outputs:

- Model: `Qwen/Qwen2-VL-7B-Instruct`
- Adapter: none; base-model zero-shot inference
- Test split: 124 samples
- Quantization: 4-bit NF4
- Decoding: `do_sample = False`, `repetition_penalty = 1.1`, `max_new_tokens = 2048`
- Test inference: 124 / 124 successful
- Format compliance: P1 = 0%, P2 = 0%, P3 = 0%, full format compliance = 0.0%
- Output file in the original run: `predictions_test_zeroshot.json`


## 1. Configure repository paths

This cell locates the repository root, points to the main WA >= 4.5 VLM test split, and defines the zero-shot prediction output path.


In [ ]:
import json
import os
from pathlib import Path

# Set CHARTNARRATOR_ROOT when running outside the repository root.
# Example: os.environ["CHARTNARRATOR_ROOT"] = "/content/ChartNarrator_public"
def resolve_project_root():
    env_root = os.environ.get("CHARTNARRATOR_ROOT")
    if env_root:
        return Path(env_root).resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd] + list(cwd.parents):
        if (candidate / "data").exists() or (candidate / "scripts").exists():
            return candidate
    return cwd

PROJECT_ROOT = resolve_project_root()

DATA_DIR = PROJECT_ROOT / "data" / "finetune_dataset_4_5"
IMAGE_DIR = DATA_DIR / "images"
TEST_FILE = DATA_DIR / "test.json"
PREDICTION_OUTPUT = PROJECT_ROOT / "data" / "evaluation" / "predictions" / "predictions_test_zeroshot.json"

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATA_DIR}")
print(f"Image dir    : {IMAGE_DIR}")
print(f"Test file    : {TEST_FILE}")
print(f"Predictions  : {PREDICTION_OUTPUT}")

if TEST_FILE.exists():
    with open(TEST_FILE, encoding="utf-8") as f:
        test_data_preview = json.load(f)
    print(f"test.json: {len(test_data_preview)} entries")
else:
    print("test.json: missing")
print(f"images: {len(list(IMAGE_DIR.glob('*.png'))) if IMAGE_DIR.exists() else 0} PNG files")


## 2. Verify or install the inference environment

This cell checks the expected PyTorch, Transformers, and Accelerate versions for Qwen2-VL inference. Installation is only triggered when the stack is missing or incompatible.


In [ ]:
import importlib
import os
import sys


def check_versions():
    """Return True when the required zero-shot inference stack is already installed."""
    try:
        import torch
        import transformers
        import accelerate
        ok = (
            torch.__version__.startswith("2.5.1")
            and transformers.__version__ == "4.46.1"
            and accelerate.__version__ == "1.0.1"
        )
        if ok:
            print("Inference environment is ready; installation skipped.")
            print(
                f"PyTorch {torch.__version__} | "
                f"Transformers {transformers.__version__} | "
                f"Accelerate {accelerate.__version__}"
            )
            return True
    except ImportError:
        pass
    return False


if check_versions():
    pass
else:
    print("Removing incompatible packages...")
    os.system("pip uninstall -y torch torchvision torchaudio accelerate transformer-engine flash-attn")

    print("Installing PyTorch 2.5.1...")
    os.system(
        "pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 "
        "--index-url https://download.pytorch.org/whl/cu121 -q"
    )

    print("Installing inference dependencies...")
    os.system(
        'pip install "transformers==4.46.1" "accelerate==1.0.1" "peft==0.11.1" '
        '"bitsandbytes==0.43.3" "qwen-vl-utils" "numpy<2.0" -q'
    )

    print("Installation finished. Restart the runtime, then rerun the notebook.")


## 3. Inspect one test example

This optional sanity check previews the first test prompt, reference response, and image field before running zero-shot inference.


In [ ]:
import json

with open(TEST_FILE, encoding="utf-8") as f:
    test_data = json.load(f)

e = test_data[0]
user_prompt = e["conversations"][0]["value"]

print("=== first test prompt preview (first 300 chars) ===")
print(user_prompt[:300])
print()
print("Contains [CHART CONTEXT]:", "[CHART CONTEXT]" in user_prompt)
print()
print("=== reference target preview (first 200 chars) ===")
print(e["conversations"][1]["value"][:200])
print()
print("=== image field ===")
print(e.get("image", "NO IMAGE FIELD"))


## 4. Normalize test image paths

This cell resolves test-split image paths under `data/finetune_dataset_4_5/images`. It does not change the text prompt used at inference, because `<image>` is stripped before generation.


In [ ]:
import json
import os

with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_data = json.load(f)

fixed_images = 0
missing_images = 0
added_image_tokens = 0

for entry in test_data:
    img_rel = entry.get("image", "")
    fname = os.path.basename(img_rel.replace("\\", "/"))
    img_abs = IMAGE_DIR / fname
    if entry.get("image") != str(img_abs):
        entry["image"] = str(img_abs)
        fixed_images += 1

    if not img_abs.exists():
        missing_images += 1

    convs = entry.get("conversations", [])
    if convs and convs[0].get("from") in ["user", "human"]:
        if "<image>" not in convs[0].get("value", ""):
            convs[0]["value"] = "<image>" + convs[0]["value"]
            added_image_tokens += 1

with open(TEST_FILE, "w", encoding="utf-8") as f:
    json.dump(test_data, f, indent=2, ensure_ascii=False)

status = "OK" if missing_images == 0 else "WARN"
print(
    f"[{status}] test.json: {len(test_data)} entries | "
    f"fixed image paths: {fixed_images} | added <image>: {added_image_tokens} | "
    f"missing images: {missing_images}"
)


## 5. Run base-model zero-shot inference

This long-running cell loads the base Qwen2-VL model in 4-bit NF4 without any LoRA adapter, applies the Qwen2-VL chat template, and writes zero-shot predictions for the test split.


In [ ]:
!pip install qwen-vl-utils -q

import gc
import json
import os

import torch
from qwen_vl_utils import process_vision_info
from tqdm import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, GenerationConfig, Qwen2VLForConditionalGeneration

PREDICTION_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("ChartNarrator Stage 2 zero-shot VLM inference")
print("=" * 70)
print(f"Test set : {TEST_FILE}")
print(f"Output   : {PREDICTION_OUTPUT}")
print("=" * 70)

gc.collect()
torch.cuda.empty_cache()

print("Loading base model in 4-bit NF4 without a LoRA adapter...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-7B-Instruct",
    quantization_config=bnb_config,
    device_map="cuda",
)
model.eval()

model.generation_config = GenerationConfig(
    bos_token_id=151643,
    eos_token_id=151645,
    pad_token_id=151643,
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-7B-Instruct")
print("Base model loaded without LoRA.")

with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_data = json.load(f)
print(f"Test samples: {len(test_data)}")

predictions = []

for idx, entry in enumerate(tqdm(test_data, desc="Zero-shot inference")):
    try:
        img_abs = entry["image"]
        user_query = entry["conversations"][0]["value"].replace("<image>", "").strip()

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img_abs},
                    {"type": "text", "text": user_query},
                ],
            }
        ]

        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=2048,
                do_sample=False,
                repetition_penalty=1.1,
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]

        has_p1 = "── Paragraph 1" in output_text
        has_p2 = "── Paragraph 2" in output_text
        has_p3 = "── Paragraph 3" in output_text

        predictions.append(
            {
                "id": entry["id"],
                "image": entry["image"],
                "morphology_family": entry.get("morphology_family", ""),
                "route_label": entry.get("route_label", ""),
                "conversations": [
                    {"from": "user", "value": user_query},
                    {"from": "assistant", "value": output_text},
                ],
                "reference": entry["conversations"][1]["value"],
                "format_check": {"p1": has_p1, "p2": has_p2, "p3": has_p3},
            }
        )

        if idx == 0:
            print("\n" + "=" * 70)
            print(f"First sample preview ({entry['id']})")
            print("=" * 70)
            print(output_text[:600])
            print("=" * 70 + "\n")

    except Exception as e:
        print(f"Sample {idx} ({entry.get('id', '?')}) failed: {e}")
        predictions.append({"id": entry.get("id", ""), "error": str(e)})

with open(PREDICTION_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)

successful = [p for p in predictions if "error" not in p]
print("\n" + "=" * 70)
print("Zero-shot inference statistics")
print("=" * 70)
print(f"Success: {len(successful)} / {len(predictions)}")
if successful:
    p1 = sum(1 for p in successful if p["format_check"]["p1"]) / len(successful) * 100
    p2 = sum(1 for p in successful if p["format_check"]["p2"]) / len(successful) * 100
    p3 = sum(1 for p in successful if p["format_check"]["p3"]) / len(successful) * 100
    all_fmt = sum(1 for p in successful if all(p["format_check"].values())) / len(successful) * 100
    print(f"Format: P1={p1:.0f}% | P2={p2:.0f}% | P3={p3:.0f}% | full={all_fmt:.1f}%")
    print("Expected format compliance is near 0% because the base model has not seen the three-paragraph target format.")
print(f"Results saved to: {PREDICTION_OUTPUT}")
